# PagedAttention & Block-Based Memory Tutorial

This notebook will help you learn exactly why vLLM's PagedAttention was a breakthrough. 

We are going to simulate the two ways of managing KV Cache memory:
1. **The Old Way (Contiguous Allocation):** Pre-allocating maximum memory. This causes massive fragmentation.
2. **The vLLM Way (Paged Allocation):** Using an OS-style Block Manager and a Block Table to allocate memory on demand.

Let's import our pure-Python simulator!

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
from llm_serving.paged_attention import BlockAllocator, simulated_paged_attention

print("Simulator loaded successfully.")

## 1. The Problem: Contiguous Memory Waste

Before PagedAttention, frameworks like standard PyTorch or HuggingFace had to allocate a single, massive contiguous tensor for the KV cache because matrix multiplication (`torch.matmul`) requires data to be physically next to each other in RAM.

Since the framework didn't know how long the user's response would be, it had to reserve the **Maximum Sequence Length** (e.g., 2048 tokens) the moment the request arrived.

In [ ]:
# Simulate the Old Way
MAX_SEQ_LEN = 2048
HEAD_DIM = 128
NUM_CONCURRENT_USERS = 4

# The system pre-allocates contiguous memory for all 4 users up to MAX_SEQ_LEN
contiguous_cache = torch.zeros((NUM_CONCURRENT_USERS, MAX_SEQ_LEN, HEAD_DIM))
print(f"Reserved Total Slots: {contiguous_cache.shape[0] * contiguous_cache.shape[1]}")

# Now imagine the actual responses finish early:
# User 0 asks a simple question, gets a 50-token answer.
# User 1 gets a 120-token answer.
# User 2 gets a 300-token answer.
# User 3 gets a 10-token answer.
actual_lengths = [50, 120, 300, 10]

total_used = sum(actual_lengths)
total_reserved = NUM_CONCURRENT_USERS * MAX_SEQ_LEN
wasted_slots = total_reserved - total_used

print(f"\nActual Tokens Generated: {total_used}")
print(f"Wasted Tokens (Internal Fragmentation): {wasted_slots}")
print(f"Memory Wasted: {(wasted_slots / total_reserved) * 100:.1f}%")

> **Observation:** We wasted 94% of the GPU memory! Because the memory *had* to be contiguous, we couldn't reuse the empty space for new users. If a 5th user arrives, the system throws an Out-Of-Memory (OOM) error, even though the GPU is 94% empty.

---

## 2. The Solution: Paged KV Cache

PagedAttention fixes this by breaking the KV cache into fixed-size **Blocks** (usually 16 tokens per block). 
Memory is no longer contiguous per user. Instead, the system maintains a giant pool of blocks and hands them out one-by-one as tokens are generated.

Let's initialize our OS-style Block Allocator.

In [ ]:
BLOCK_SIZE = 16
NUM_BLOCKS = 100 # A small physical pool of 100 blocks (1600 tokens total capacity)

allocator = BlockAllocator(num_blocks=NUM_BLOCKS, block_size=BLOCK_SIZE, head_dim=HEAD_DIM)
print(f"Initialized Physical Pool with {len(allocator.free_blocks)} free blocks.")

### Simulating Dynamic On-Demand Allocation
Let's simulate User 0 generating 50 tokens. We will append them one by one, just like a real autoregressive decode loop.

In [ ]:
seq_id = "User_0"

print("Generating tokens for User 0...")
for step in range(50):
    # Create a dummy token KV vector
    dummy_k = torch.randn(HEAD_DIM)
    dummy_v = torch.randn(HEAD_DIM)
    
    # The allocator will automatically grab a new physical block ONLY when the current one is full (every 16 tokens)
    allocator.append_token(seq_id, dummy_k, dummy_v)

print("\n--- Block Table Mapping ---")
# Notice how the blocks don't have to be contiguous! 
# User 0 might own Physical Blocks [0, 1, 2, 3]
print(f"{seq_id}'s Logical-to-Physical Block Table: {allocator.block_tables[seq_id]}")
print(f"Tokens generated: {allocator.seq_lengths[seq_id]}")

stats = allocator.get_fragmentation_stats()
print(f"\nInternal Fragmentation (Wasted Slots): {stats['wasted_slots']} tokens ({stats['internal_fragmentation_pct']:.1f}%)")

> **Observation:** User 0 generated 50 tokens. Because each block holds 16 tokens, the OS gave User 0 exactly 4 physical blocks (capacity: 64 tokens). 
> 
> Instead of reserving 2048 tokens and wasting 1998 slots, we only wasted **14 slots** (the empty space at the end of the 4th block). **This eliminates external fragmentation and strictly limits internal fragmentation to < 1 block.**

## 3. The Magic: Paged Attention Math

If the memory is no longer contiguous (User 1's memory might be scattered across Block 5, Block 12, and Block 99), how do we do the Attention math (`Query * Key^T`)?

This is what the `PagedAttention` custom CUDA kernel does. It iterates through the Block Table, fetches the physical blocks one by one, computes the attention scores for that chunk, and stitches them together.

Let's run our pure-Python simulation of that math.

In [ ]:
# The model generates a new Query vector for the next token
new_query = torch.randn((1, HEAD_DIM))

# We pass the Query, the Sequence ID, and the Allocator to our PagedAttention simulator
# It will look up User 0's Block Table, gather the scattered blocks, and compute attention.
attention_output = simulated_paged_attention(new_query, "User_0", allocator)

print(f"Query shape: {new_query.shape}")
print(f"Attention Output shape: {attention_output.shape}")
print("\nSuccess! We computed attention across non-contiguous physical memory.")

## Summary

By playing with this simulator, you should now understand:
1. **Why standard serving OOMs so easily:** Pre-allocating `max_seq_len` wastes massive amounts of memory due to unknown output lengths.
2. **How Block Tables work:** By mapping logical blocks to physical blocks, sequences can grow dynamically one block (16 tokens) at a time.
3. **How PagedAttention solves the math:** It abstracts the fragmented physical memory away from the Attention mechanism by fetching scattered blocks on the fly.

**In your project (Topic 2):**
PagedAttention solves *fragmentation* (wasted space). But it doesn't change the *size* of the actual data. 
If you have 10,000 blocks filled with FP16 data, you will still run out of memory. This is why you must combine PagedAttention with **FP8 Quantization**—FP8 shrinks the size of the blocks themselves by 50%, letting you store twice as many blocks in the same physical GPU.